In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Threat_Industry_DS.csv')

print('Dataset Shape:')
print(df.shape)
print('\n')

print('First 5 rows of the dataset:')
print(df.head())
print('\n')

print('Count of null values per column:')
print(df.isnull().sum())

Dataset Shape:
(1592, 3)


First 5 rows of the dataset:
         date          country                              industry
0  14/11/2025              USA                Healthcare and Medical
1   5/11/2025  Morocco, France                 Retail and E-commerce
2   5/11/2025              USA  Consulting and Professional Services
3  29/10/2025  Spain, Portugal                Healthcare and Medical
4  22/10/2025              USA                     Security Services


Count of null values per column:
date         80
country     418
industry      0
dtype: int64


In [6]:
# 1. Replace any empty country cells with "USA"
df['country'].fillna('USA', inplace=True)

# 2. Replace any of the empty dates with "09/2020"
df['date'].fillna('09/2020', inplace=True)

# 3. Format the date column from 'DD/MM/YYYY' to 'MM/YYYY'
#    The filled '09/2020' is already in 'MM/YYYY' format, so we only transform dates in 'DD/MM/YYYY' format.
import re

def reformat_date(date_str):
    date_str = str(date_str)

    # Regular expression to match various forms of D/M/YYYY (e.g., 5/11/2025, 14/1/2025)
    day_month_year_pattern = re.compile(r'^\d{1,2}/\d{1,2}/\d{4}$|^\d{1,2}/\d{4}$|^\d{4}$')

    if day_month_year_pattern.match(date_str):
        try:
            # Try parsing with dayfirst=True to handle DD/MM/YYYY, D/MM/YYYY, etc.
            # The errors='coerce' will turn unparseable dates into NaT, which we can then filter.
            # However, for this specific logic, we want to keep original if it doesn't match expected formats for transformation.
            # Let's try parsing directly assuming day is first.
            dt_object = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
            if pd.notna(dt_object):
                return dt_object.strftime('%m/%Y')
            else:
                # If pd.to_datetime with dayfirst=True failed, check if it's already MM/YYYY
                if re.fullmatch(r'\d{1,2}/\d{4}', date_str): # e.g., '09/2020'
                    return date_str
                elif re.fullmatch(r'\d{4}', date_str): # e.g., '2025'
                    return date_str
                else:
                    return date_str # Return original if still unparseable and not MM/YYYY or YYYY
        except ValueError:
            return date_str # Fallback for unexpected parsing errors
    elif re.fullmatch(r'\d{1,2}/\d{4}', date_str) or re.fullmatch(r'\d{4}', date_str):
        # If it's already in MM/YYYY format (like '09/2020') or YYYY format
        return date_str
    else:
        # If it doesn't match any known pattern, keep it as is
        return date_str

df['date'] = df['date'].apply(reformat_date)

# Display some info after transformations to verify
print("\nDataset after transformations (first 5 rows):")
print(df.head())
print('\n')
print("Null values after transformations:")
print(df.isnull().sum())
print('\n')
print("Unique dates after transformations (first 10 unique values):")
print(df['date'].unique()[:10])

/tmp/ipython-input-271232751.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna('USA', inplace=True)
/tmp/ipython-input-271232751.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.m


Dataset after transformations (first 5 rows):
      date country                              industry
0  11/2025     Usa                Healthcare and Medical
1  11/2025  France                 Retail and E-commerce
2  11/2025     Usa  Consulting and Professional Services
3  10/2025   Spain                Healthcare and Medical
4  10/2025     Usa                     Security Services


Null values after transformations:
date        0
country     0
industry    0
dtype: int64


Unique dates after transformations (first 10 unique values):
['11/2025' '10/2025' '09/2025' '08/2025' '07/2025' '06/2025' '05/2025'
 '04/2025' '03/2025' '02/2025']


In [7]:
import re

def clean_and_prioritize_country(country_string):
    if not isinstance(country_string, str): # Should not happen after previous fillna, but for robustness
        return country_string

    # Split by common delimiters (comma, 'and') and clean up each part
    # Convert to title case for consistency (e.g., 'usa' becomes 'Usa')
    # filter out empty strings after stripping
    countries_list = [c.strip().title() for c in re.split(r',| and ', country_string) if c.strip()]

    # Normalize USA variations and collect non-USA countries
    normalized_countries = []
    non_usa_countries = []

    for country in countries_list:
        if country in ['United States', 'America']:
            normalized_countries.append('USA')
        else:
            normalized_countries.append(country)
            if country != 'USA': # Explicitly check for 'USA' in title case
                non_usa_countries.append(country)

    # Remove duplicates while maintaining order implicitly for prioritization
    unique_normalized_countries = []
    for country in normalized_countries:
        if country not in unique_normalized_countries:
            unique_normalized_countries.append(country)

    # Prioritize non-USA countries
    if non_usa_countries:
        # Return the first unique non-USA country found
        return list(set(non_usa_countries))[0] # Using set to get unique, then take first
    elif 'USA' in unique_normalized_countries:
        # If only USA or its variations were present
        return 'USA'
    else:
        # Fallback if no valid country names were found (e.g., empty string after cleaning)
        return '' # Or None, or 'Unknown'

# Apply the function to the 'country' column
df['country'] = df['country'].apply(clean_and_prioritize_country)

print("\nDataset after country cleaning and prioritization (first 5 rows):")
print(df.head())
print('\n')
print("Unique countries and their counts after transformations:")
print(df['country'].value_counts())



Dataset after country cleaning and prioritization (first 5 rows):
      date country                              industry
0  11/2025     Usa                Healthcare and Medical
1  11/2025  France                 Retail and E-commerce
2  11/2025     Usa  Consulting and Professional Services
3  10/2025   Spain                Healthcare and Medical
4  10/2025     Usa                     Security Services


Unique countries and their counts after transformations:
country
Usa               1166
Germany             53
United Kingdom      32
Canada              31
Russia              27
                  ... 
Barbados             1
Croatia              1
Greece               1
Finland              1
Jamaica              1
Name: count, Length: 100, dtype: int64


In [8]:
df.head()

,date,country,industry
0,11/2025,Usa,Healthcare and Medical
1,11/2025,France,Retail and E-commerce
2,11/2025,Usa,Consulting and Professional Services
3,10/2025,Spain,Healthcare and Medical
4,10/2025,Usa,Security Services


In [9]:
def clean_text_column(text):
    if isinstance(text, str):
        # Convert to lowercase
        text = text.lower()
        # Remove periods and commas
        text = text.replace('.', '').replace(',', '')
    return text

# Apply the cleaning function to 'country' and 'industry' columns
df['country'] = df['country'].apply(clean_text_column)
df['industry'] = df['industry'].apply(clean_text_column)

print("\nDataset after country and industry cleaning (first 5 rows):")
print(df.head())
print('\n')
print("Unique countries and their counts after transformations:")
print(df['country'].value_counts().head(10))
print('\n')
print("Unique industries and their counts after transformations:")
print(df['industry'].value_counts().head(10))


Dataset after country and industry cleaning (first 5 rows):
      date country                              industry
0  11/2025     usa                healthcare and medical
1  11/2025  france                 retail and e-commerce
2  11/2025     usa  consulting and professional services
3  10/2025   spain                healthcare and medical
4  10/2025     usa                     security services


Unique countries and their counts after transformations:
country
usa               1166
germany             53
united kingdom      32
canada              31
russia              27
italy               18
israel              14
australia           11
brazil              10
spain               10
Name: count, dtype: int64


Unique industries and their counts after transformations:
industry
consulting and professional services    217
government and public sector            187
telecommunications                      173
technology and software                 145
manufacturing and industrial 

In [10]:
# Save the updated DataFrame to a new CSV file
df.to_csv('threat_inndustry_v1.csv', index=False)

print("DataFrame saved successfully as 'threat_inndustry_v1.csv'")

DataFrame saved successfully as 'threat_inndustry_v1.csv'
